In [1]:
"""
Smart Personal Finance Tracker
Single-file reference implementation (Python 3.9+)

Features:
- Log & categorize income/expenses
- Set budget limits and alerts when nearing limits
- Visualize trends (monthly/yearly) using matplotlib
- Store data securely using SQLite (local file)
- Suggest optimized monthly budgets via greedy & backtracking
- CLI interface
- Stubs for ChatGPT-like assistant integration

How to run:
1. Install dependencies: pip install matplotlib pandas
2. Run: python smart_finance_tracker.py

Note: This is a modular but single-file demo to make it easy to run and iterate.
For production split into modules, add unit tests, and consider SQLAlchemy + encryption for the DB file.
"""

import sqlite3
import datetime
import logging
import os
import sys
import argparse
import json
from collections import defaultdict
from typing import List, Dict, Optional, Tuple

# Visualization and data
import matplotlib.pyplot as plt
import pandas as pd

# ---------------------
# Logging configuration
# ---------------------
LOG_FILE = 'finance_tracker.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger('FinanceTracker')

# ---------------------
# Database layer
# ---------------------
DB_FILE = 'finance_tracker.db'

class Database:
    def __init__(self, path: str = DB_FILE):
        self.path = path
        self.conn = None
        self._connect()
        self._migrate()

    def _connect(self):
        try:
            self.conn = sqlite3.connect(self.path, check_same_thread=False)
            self.conn.row_factory = sqlite3.Row
            logger.info(f"Connected to DB at {self.path}")
        except sqlite3.Error as e:
            logger.exception("Failed connecting to DB")
            raise

    def _migrate(self):
        try:
            cur = self.conn.cursor()
            # transactions table
            cur.execute('''
            CREATE TABLE IF NOT EXISTS transactions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                date TEXT NOT NULL,
                amount REAL NOT NULL,
                category TEXT NOT NULL,
                type TEXT NOT NULL CHECK(type IN ('expense','income')),
                note TEXT
            );
            ''')
            # budgets table
            cur.execute('''
            CREATE TABLE IF NOT EXISTS budgets (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                year INTEGER NOT NULL,
                month INTEGER NOT NULL,
                category TEXT NOT NULL,
                limit_amount REAL NOT NULL,
                UNIQUE(year, month, category)
            );
            ''')
            # alerts table (simple history)
            cur.execute('''
            CREATE TABLE IF NOT EXISTS alerts (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                date TEXT NOT NULL,
                message TEXT NOT NULL
            );
            ''')
            self.conn.commit()
            logger.info('Database migrated (tables ready)')
        except sqlite3.Error:
            logger.exception('Migration failed')
            raise

    def execute(self, query: str, params: tuple = (), commit: bool = False):
        try:
            cur = self.conn.cursor()
            cur.execute(query, params)
            if commit:
                self.conn.commit()
            return cur
        except sqlite3.Error:
            logger.exception('DB query failed')
            raise

    def close(self):
        try:
            if self.conn:
                self.conn.close()
                logger.info('DB closed')
        except Exception:
            logger.exception('Error closing DB')

# ---------------------
# Transaction manager
# ---------------------
class TransactionManager:
    def __init__(self, db: Database):
        self.db = db

    def add_transaction(self, date: str, amount: float, category: str, ttype: str, note: Optional[str] = None) -> int:
        assert ttype in ('expense','income')
        try:
            cur = self.db.execute(
                'INSERT INTO transactions (date, amount, category, type, note) VALUES (?, ?, ?, ?, ?)',
                (date, amount, category, ttype, note), commit=True
            )
            tid = cur.lastrowid
            logger.info(f'Added transaction {tid} {ttype} {amount} {category} {date}')
            return tid
        except Exception:
            logger.exception('Failed to add transaction')
            raise

    def get_transactions(self, start_date: Optional[str] = None, end_date: Optional[str] = None, category: Optional[str] = None) -> List[sqlite3.Row]:
        q = 'SELECT * FROM transactions WHERE 1=1 '
        params = []
        if start_date:
            q += ' AND date >= ?'
            params.append(start_date)
        if end_date:
            q += ' AND date <= ?'
            params.append(end_date)
        if category:
            q += ' AND category = ?'
            params.append(category)
        q += ' ORDER BY date ASC'
        cur = self.db.execute(q, tuple(params))
        return cur.fetchall()

    def monthly_summary(self, year: int, month: int) -> Dict[str, float]:
        start = f"{year:04d}-{month:02d}-01"
        # compute end date as first of next month minus one day
        if month == 12:
            end = f"{year+1:04d}-01-01"
        else:
            end = f"{year:04d}-{month+1:02d}-01"
        cur = self.db.execute(
            "SELECT type, category, SUM(amount) as total FROM transactions WHERE date >= ? AND date < ? GROUP BY type, category",
            (start, end)
        )
        rows = cur.fetchall()
        res = defaultdict(float)
        for r in rows:
            key = f"{r['type']}::{r['category']}"
            res[key] = r['total']
        return res

# ---------------------
# Budget manager
# ---------------------
class BudgetManager:
    ALERT_THRESHOLD = 0.8  # 80%

    def __init__(self, db: Database, tx_mgr: TransactionManager):
        self.db = db
        self.tx = tx_mgr

    def set_budget(self, year: int, month: int, category: str, limit_amount: float):
        try:
            self.db.execute(
                'INSERT OR REPLACE INTO budgets (year, month, category, limit_amount) VALUES (?, ?, ?, ?)',
                (year, month, category, limit_amount), commit=True
            )
            logger.info(f'Set budget for {year}-{month} {category} = {limit_amount}')
        except Exception:
            logger.exception('Failed to set budget')
            raise

    def get_budgets(self, year: int, month: int) -> Dict[str, float]:
        cur = self.db.execute('SELECT category, limit_amount FROM budgets WHERE year = ? AND month = ?', (year, month))
        return {row['category']: row['limit_amount'] for row in cur.fetchall()}

    def check_alerts(self, year: int, month: int) -> List[str]:
        budgets = self.get_budgets(year, month)
        alerts = []
        for cat, limit in budgets.items():
            summary = self.tx.monthly_summary(year, month)
            spent = summary.get(f'expense::{cat}', 0.0)
            if limit > 0 and spent / limit >= self.ALERT_THRESHOLD:
                msg = f"Alert: spending for {cat} is at {spent:.2f}/{limit:.2f} ({spent/limit:.0%})"
                alerts.append(msg)
                # store alert history
                self.db.execute('INSERT INTO alerts (date, message) VALUES (?, ?)', (datetime.date.today().isoformat(), msg), commit=True)
                logger.info('Alert generated: ' + msg)
        return alerts

# ---------------------
# Visualizer
# ---------------------
class Visualizer:
    def __init__(self, db: Database):
        self.db = db

    def _load_transactions_df(self, start_date: Optional[str] = None, end_date: Optional[str] = None) -> pd.DataFrame:
        q = 'SELECT date, amount, category, type FROM transactions WHERE 1=1'
        params = []
        if start_date:
            q += ' AND date >= ?'
            params.append(start_date)
        if end_date:
            q += ' AND date <= ?'
            params.append(end_date)
        cur = self.db.execute(q + ' ORDER BY date')
        rows = cur.fetchall()
        if not rows:
            return pd.DataFrame(columns=['date','amount','category','type'])
        df = pd.DataFrame([dict(r) for r in rows])
        df['date'] = pd.to_datetime(df['date'])
        return df

    def plot_monthly_category_trend(self, category: str, months: int = 12):
        today = datetime.date.today()
        start_month = (today.replace(day=1) - pd.DateOffset(months=months-1)).date()
        df = self._load_transactions_df(start_date=start_month.isoformat())
        if df.empty:
            print('No data to plot')
            return
        df = df[df['category'] == category]
        if df.empty:
            print(f'No data for category: {category}')
            return
        df_exp = df[df['type'] == 'expense']
        df_exp['ym'] = df_exp['date'].dt.to_period('M')
        grouped = df_exp.groupby('ym')['amount'].sum().sort_index()
        grouped.plot(kind='bar')
        plt.title(f'Monthly spending for \"{category}\"')
        plt.xlabel('Month')
        plt.ylabel('Amount')
        plt.tight_layout()
        plt.show()

    def plot_income_vs_expense(self, year: int):
        start = f"{year:04d}-01-01"
        end = f"{year+1:04d}-01-01"
        df = self._load_transactions_df(start_date=start, end_date=end)
        if df.empty:
            print('No data to plot')
            return
        df['month'] = df['date'].dt.month
        monthly = df.groupby(['month','type'])['amount'].sum().unstack(fill_value=0)
        monthly.plot(kind='line', marker='o')
        plt.title(f'Income vs Expense in {year}')
        plt.xlabel('Month')
        plt.ylabel('Amount')
        plt.tight_layout()
        plt.show()

# ---------------------
# Budget optimizer
# ---------------------
class BudgetOptimizer:
    """
    Two strategies:
    - Greedy: allocate budgets proportional to average past spending per category
    - Backtracking (search): try to find a budget allocation under a total cap that maximizes coverage of historically high-utility categories.
    """
    def __init__(self, db: Database, tx_mgr: TransactionManager):
        self.db = db
        self.tx = tx_mgr

    def _historic_avg_spend(self, months: int = 6) -> Dict[str, float]:
        today = datetime.date.today()
        start = (today.replace(day=1) - pd.DateOffset(months=months-1)).date().isoformat()
        cur = self.db.execute('SELECT category, SUM(amount) as total FROM transactions WHERE type = "expense" AND date >= ? GROUP BY category', (start,))
        rows = cur.fetchall()
        res = {}
        for r in rows:
            res[r['category']] = r['total'] / months
        return res

    def greedy_allocate(self, total_budget: float, months: int = 6) -> Dict[str, float]:
        avgs = self._historic_avg_spend(months=months)
        if not avgs:
            return {}
        total_avg = sum(avgs.values())
        if total_avg == 0:
            # equal split
            n = len(avgs)
            return {cat: total_budget/n for cat in avgs}
        allocation = {cat: (avg/total_avg)*total_budget for cat, avg in avgs.items()}
        logger.info('Greedy allocation computed')
        return allocation

    def backtracking_optimize(self, total_budget: float, months: int = 6, max_states: int = 20000) -> Dict[str, float]:
        # simplified discrete approach: treat each category budget as multiple of an increment
        avgs = self._historic_avg_spend(months=months)
        if not avgs:
            return {}
        cats = list(avgs.keys())
        # utility = historical average (higher spend => higher utility to fund)
        utils = [avgs[c] for c in cats]
        # discretize budget into increments
        increment = max(1.0, total_budget/100.0)
        max_units = int(total_budget // increment)

        best_val = -1
        best_alloc = None
        states = 0

        # simple DFS: allocate units per category 0..max_units (but prune)
        allocation_units = [0]*len(cats)

        def dfs(i, units_left, current_val):
            nonlocal best_val, best_alloc, states
            if states > max_states:
                return
            states += 1
            if i == len(cats):
                if current_val > best_val:
                    best_val = current_val
                    best_alloc = allocation_units.copy()
                return
            # upper bound pruning: allocate all remaining to this category
            # try allocations from 0..units_left
            for u in range(units_left, -1, -1):
                allocation_units[i] = u
                dfs(i+1, units_left - u, current_val + u*utils[i])
                allocation_units[i] = 0
                if states > max_states:
                    return

        dfs(0, max_units, 0)
        if best_alloc is None:
            # fallback to greedy
            return self.greedy_allocate(total_budget, months=months)
        res = {}
        for cat, units in zip(cats, best_alloc):
            res[cat] = units * increment
        logger.info('Backtracking optimization finished')
        return res

# ---------------------
# Small assistant (stub)
# ---------------------
class Assistant:
    """
    Very small query parser to answer questions like "How much did I spend on food last week?"

    For full ChatGPT integration: user can supply OpenAI API key and call the API. This stub uses simple heuristics.
    """
    def __init__(self, tx_mgr: TransactionManager):
        self.tx = tx_mgr

    def answer(self, question: str) -> str:
        q = question.lower()
        # parse for category
        import re
        cat_match = re.search(r'on ([a-zA-Z ]+?)(?: last| in | this| for |\?)', q)
        if cat_match:
            category = cat_match.group(1).strip()
        else:
            # fallback to common words
            tokens = q.split()
            category = None
            for t in ['food','rent','entertainment','transport','groceries','salary']:
                if t in q:
                    category = t
                    break
        # parse for time window
        if 'last week' in q:
            today = datetime.date.today()
            start = (today - datetime.timedelta(days=today.weekday()+7)).isoformat()  # start of last week monday
            end = (datetime.date.fromisoformat(start) + datetime.timedelta(days=7)).isoformat()
        elif 'this month' in q:
            today = datetime.date.today()
            start = today.replace(day=1).isoformat()
            end = (today + pd.DateOffset(months=1)).date().isoformat()
        else:
            # default last 30 days
            end = datetime.date.today().isoformat()
            start = (datetime.date.today() - datetime.timedelta(days=30)).isoformat()

        rows = self.tx.get_transactions(start_date=start, end_date=end, category=category) if category else self.tx.get_transactions(start_date=start, end_date=end)
        total = sum(r['amount'] for r in rows if r['type']=='expense')
        return f"Spent {total:.2f} on {category or 'all categories'} between {start} and {end}"

# ---------------------
# CLI interface
# ---------------------
class CLI:
    def __init__(self):
        self.db = Database()
        self.tx = TransactionManager(self.db)
        self.budget = BudgetManager(self.db, self.tx)
        self.vis = Visualizer(self.db)
        self.opt = BudgetOptimizer(self.db, self.tx)
        self.assistant = Assistant(self.tx)

    def run(self):
        parser = argparse.ArgumentParser(prog='FinanceTracker', description='Smart Personal Finance Tracker')
        parser.add_argument('--interactive', action='store_true', help='Run interactive prompt')
        parser.add_argument('--add', nargs=4, metavar=('DATE','AMOUNT','CATEGORY','TYPE'), help='Add transaction: DATE(YYYY-MM-DD) AMOUNT CATEGORY TYPE(expense|income)')
        parser.add_argument('--set-budget', nargs=3, metavar=('YYYY-MM','CATEGORY','AMOUNT'), help='Set budget for month')
        parser.add_argument('--check-alerts', action='store_true', help='Check alerts for current month')
        parser.add_argument('--plot-category', nargs=1, metavar=('CATEGORY',), help='Plot monthly trend for CATEGORY')
        parser.add_argument('--plot-year', nargs=1, metavar=('YEAR',), type=int, help='Plot income vs expense for YEAR')
        parser.add_argument('--suggest-budget', nargs=1, metavar=('TOTAL',), type=float, help='Suggest budgets given total amount (greedy & backtracking)')
        parser.add_argument('--ask', nargs='+', help='Ask the assistant a question (simple queries supported)')
        args = parser.parse_args()

        try:
            if args.interactive:
                self._interactive_prompt()
            elif args.add:
                date, amount, category, ttype = args.add
                self.tx.add_transaction(date, float(amount), category, ttype)
            elif args.set_budget:
                ym, category, amount = args.set_budget
                year, month = map(int, ym.split('-'))
                self.budget.set_budget(year, month, category, float(amount))
            elif args.check_alerts:
                today = datetime.date.today()
                alerts = self.budget.check_alerts(today.year, today.month)
                if alerts:
                    for a in alerts:
                        print(a)
                else:
                    print('No alerts')
            elif args.plot_category:
                self.vis.plot_monthly_category_trend(args.plot_category[0])
            elif args.plot_year:
                self.vis.plot_income_vs_expense(args.plot_year[0])
            elif args.suggest_budget:
                total = args.suggest_budget[0]
                greedy = self.opt.greedy_allocate(total)
                back = self.opt.backtracking_optimize(total)
                print('\nGreedy allocation:')
                print(json.dumps(greedy, indent=2))
                print('\nBacktracking allocation:')
                print(json.dumps(back, indent=2))
            elif args.ask:
                q = ' '.join(args.ask)
                ans = self.assistant.answer(q)
                print(ans)
            else:
                parser.print_help()
        except Exception as e:
            logger.exception('Error during CLI operation')
            print('Operation failed:', e)

    def _interactive_prompt(self):
        print('Welcome to Smart Finance Tracker (interactive mode). Type "help" for commands.')
        while True:
            try:
                cmd = input('> ').strip()
                if not cmd:
                    continue
                if cmd in ('exit','quit'):
                    print('Goodbye')
                    break
                if cmd == 'help':
                    print('\nCommands: add, set-budget, check-alerts, plot-category, plot-year, suggest-budget, ask, quit')
                    continue
                parts = cmd.split()
                c = parts[0]
                if c == 'add':
                    # add 2025-08-10 250 food expense lunch
                    if len(parts) < 5:
                        print('Usage: add DATE(YYYY-MM-DD) AMOUNT CATEGORY TYPE [note]')
                        continue
                    date, amount, category, ttype = parts[1:5]
                    note = ' '.join(parts[5:]) if len(parts) > 5 else None
                    self.tx.add_transaction(date, float(amount), category, ttype, note)
                elif c == 'set-budget':
                    # set-budget 2025-08 food 5000
                    if len(parts) != 4:
                        print('Usage: set-budget YYYY-MM CATEGORY AMOUNT')
                        continue
                    ym, category, amount = parts[1:4]
                    year, month = map(int, ym.split('-'))
                    self.budget.set_budget(year, month, category, float(amount))
                elif c == 'check-alerts':
                    today = datetime.date.today()
                    alerts = self.budget.check_alerts(today.year, today.month)
                    if alerts:
                        for a in alerts:
                            print(a)
                    else:
                        print('No alerts')
                elif c == 'plot-category':
                    if len(parts) != 2:
                        print('Usage: plot-category CATEGORY')
                        continue
                    self.vis.plot_monthly_category_trend(parts[1])
                elif c == 'plot-year':
                    if len(parts) != 2:
                        print('Usage: plot-year YYYY')
                        continue
                    self.vis.plot_income_vs_expense(int(parts[1]))
                elif c == 'suggest-budget':
                    if len(parts) != 2:
                        print('Usage: suggest-budget TOTAL')
                        continue
                    total = float(parts[1])
                    greedy = self.opt.greedy_allocate(total)
                    back = self.opt.backtracking_optimize(total)
                    print('\nGreedy allocation:')
                    print(json.dumps(greedy, indent=2))
                    print('\nBacktracking allocation:')
                    print(json.dumps(back, indent=2))
                elif c == 'ask':
                    q = ' '.join(parts[1:])
                    print(self.assistant.answer(q))
                else:
                    print('Unknown command. Type help.')
            except Exception:
                logger.exception('Interactive command failed')
                print('Error executing command')

# ---------------------
# Entry point
# ---------------------
if __name__ == '__main__':
    CLI().run()


2025-11-22 23:14:37,720 [INFO] Connected to DB at finance_tracker.db
2025-11-22 23:14:37,722 [INFO] Database migrated (tables ready)


usage: FinanceTracker [-h] [--interactive] [--add DATE AMOUNT CATEGORY TYPE]
                      [--set-budget YYYY-MM CATEGORY AMOUNT] [--check-alerts]
                      [--plot-category CATEGORY] [--plot-year YEAR]
                      [--suggest-budget TOTAL] [--ask ASK [ASK ...]]
FinanceTracker: error: unrecognized arguments: --f=c:\Users\athar\AppData\Roaming\jupyter\runtime\kernel-v3b927fda60edd2ec4b64e94289fc17eeea83cccb7.json


SystemExit: 2

C:\Users\athar\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
